03 - H4: Rebranding statt Buzzword-Inflation.

Befund: Die ursprueglich vermutete Buzzword-Inflation laesst sich datenseitig
nicht belegen. Stattdessen zeigt sich politisches Rebranding - vor allem das
vollstaendige Verschwinden der Hightech-Strategie (Merkel-Aera) und das
Erscheinen der Zukunftsstrategie (Ampel) bei vergleichbarem Volumen.

Drilldowns:
  1. ZEW-Schlagwoerter aus titel_text extrahieren (eckige Klammern)
  2. Treffer- und Volumen-Trends 2019 vs. 2024 je Schlagwort
  3. Wachstumsvergleich n vs. Volumen (Inflations-Indikator)

Outputs: figures/h4_rebranding_slope.png

In [1]:
import sys
import re
from pathlib import Path
from collections import Counter
_root = (
    Path(__file__).parent.parent if '__file__' in globals() else
    next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src').exists())
)
sys.path.insert(0, str(_root))
__import__('os').chdir(_root)

import pandas as pd
from src.load import load

df = load('data/raw/Digitalhaushalt_Open_Data.xlsx')


def clean_tag(s):
    """Tag normalisieren: geschweifte Sub-Marker raus, lowercase, trim."""
    s = re.sub(r'\{|\}', '', s).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s


def extract_tags(text):
    if not isinstance(text, str):
        return []
    return [clean_tag(t) for t in re.findall(r'\[([^\]]+)\]', text)]

In [2]:
df['tags'] = df['titel_text'].apply(extract_tags)
print(f"Titel mit mindestens einem ZEW-Tag: "
      f"{df['tags'].apply(len).gt(0).sum():,} von {len(df):,}")

Titel mit mindestens einem ZEW-Tag: 3,784 von 21,358


In [3]:
all_tags = Counter()
for t in df['tags']:
    all_tags.update(t)

print("\n=== Top-20 ZEW-Schlagwoerter, alle Jahre kumuliert ===")
for tag, n in all_tags.most_common(20):
    print(f"  {n:>5}  {tag}")


=== Top-20 ZEW-Schlagwoerter, alle Jahre kumuliert ===
    911  software
    414  informationstechn
    197  it
    187  netzpolitik
    164  bundesnetzagentur
    158  e mobilität
    133  bundesamt für sicherheit in der informationstechnik
    124  bundeskartellamt
    105  digitale infrastruktur
     92  informationstechnikzentrum bund
     91  zukunftsstrategie
     89  zentrale stelle für informationstechnik im sicherheitsbereich
     86  datenschutz und die informationsfreiheit
     84  it-
     84  leibniz
     84  hightech-strategie
     75  digitale
     66  digitalisierung
     47  vernetz
     44  fernmelde


In [4]:
top_tags = [t for t, _ in all_tags.most_common(25)]
res = []
for tag in top_tags:
    has_tag = df['tags'].apply(lambda lst: tag in lst)
    row = {'tag': tag}
    for j in sorted(df['jahr'].unique()):
        sub = df[has_tag & (df['jahr'] == j)]
        row[f'n_{j}'] = len(sub)
        row[f'v_{j}_Mio'] = round(sub['digi_soll_eng'].sum() / 1e3, 1)
    res.append(row)
out = pd.DataFrame(res)

In [5]:
print("\n=== Wachstum Treffer vs. Volumen 2019 -> 2024 ===")
print(f"{'Tag':<35} {'n19':>4} {'n24':>4} {'n_x':>5} "
      f"{'v19_Mio':>9} {'v24_Mio':>9} {'v_x':>6}")
print("-" * 80)
for _, r in out.iterrows():
    n19, n24 = r['n_2019'], r['n_2024']
    v19, v24 = r['v_2019_Mio'], r['v_2024_Mio']
    n_fac = f"{n24/n19:.1f}x" if n19 > 0 else 'neu'
    v_fac = f"{v24/v19:.1f}x" if v19 > 0 else 'neu'
    print(f"{r['tag'][:34]:<35} {n19:>4} {n24:>4} {n_fac:>5} "
          f"{v19:>9.0f} {v24:>9.0f} {v_fac:>6}")


=== Wachstum Treffer vs. Volumen 2019 -> 2024 ===
Tag                                  n19  n24   n_x   v19_Mio   v24_Mio    v_x
--------------------------------------------------------------------------------
software                             214  237  1.1x       466       634   1.4x
informationstechn                     98  108  1.1x      1541      3539   2.3x
it                                    43   54  1.3x      1238      1052   0.8x
netzpolitik                           41   52  1.3x      1238      1037   0.8x
bundesnetzagentur                     37   45  1.2x        37        43   1.2x
e mobilität                           29   43  1.5x       748      1067   1.4x
bundesamt für sicherheit in der in    33   34  1.0x       137       238   1.7x
bundeskartellamt                      30   32  1.1x        11        22   2.1x
digitale infrastruktur                26   25  1.0x       218      3562  16.4x
informationstechnikzentrum bund       24   22  0.9x       535      1311   2.5x

In [6]:
print("\n=== Rebranding-Befund: Politisches Etikett wechselt, Inhalt bleibt ===")
for tag in ['hightech-strategie', 'zukunftsstrategie']:
    has_tag = df['tags'].apply(lambda lst, t=tag: t in lst)
    for j in sorted(df['jahr'].unique()):
        sub = df[has_tag & (df['jahr'] == j)]
        n, v = len(sub), sub['digi_soll_eng'].sum() / 1e6
        print(f"  {tag:<22} {j}: n={n:>3}, V={v:>5.2f} Mrd")
    print()


=== Rebranding-Befund: Politisches Etikett wechselt, Inhalt bleibt ===
  hightech-strategie     2019: n= 40, V= 1.32 Mrd
  hightech-strategie     2021: n= 44, V= 1.49 Mrd
  hightech-strategie     2023: n=  0, V= 0.00 Mrd
  hightech-strategie     2024: n=  0, V= 0.00 Mrd

  zukunftsstrategie      2019: n=  0, V= 0.00 Mrd
  zukunftsstrategie      2021: n=  0, V= 0.00 Mrd
  zukunftsstrategie      2023: n= 45, V= 1.89 Mrd
  zukunftsstrategie      2024: n= 46, V= 1.89 Mrd

